# Flash Project: Employee Insights & Department Performance

This capstone project brings together the main SQL concepts from the course: joins, aggregations, CASE logic, CTEs, and window functions.

## Business scenario
A company wants a quick operational dashboard to answer key questions about employee performance, department salaries, and workforce structure.

We will use three tables: `employees`, `departments`, and `performance`.

## 1) Create the project dataset

Use temporary views so the dataset is easy to reset and rerun.

In [ ]:
CREATE OR REPLACE TEMP VIEW employees AS
SELECT * FROM VALUES
  (1, 'John', 'IT', 'Senior Engineer', 90000, '2020-01-15', 1),
  (2, 'Mary', 'HR', 'HR Manager', 78000, '2019-06-10', 0),
  (3, 'David', 'IT', 'Data Engineer', 96000, '2021-03-12', 1),
  (4, 'Lisa', 'Finance', 'Accountant', 86000, '2018-11-20', 1),
  (5, 'Robert', 'Sales', 'Sales Rep', 68000, '2022-02-05', 1),
  (6, 'Sarah', 'HR', 'Recruiter', 76000, '2021-09-18', 0),
  (7, 'Tom', NULL, 'Intern', 60000, '2023-07-01', 0),
  (8, 'Ava', 'Sales', 'Sales Manager', 97000, '2017-04-22', 1),
  (9, 'Noah', 'Finance', 'Analyst', 73000, '2022-11-30', 1),
  (10, 'Emma', 'IT', 'Developer', 82000, '2020-08-14', 1)
AS t(employee_id, name, department, role_title, salary, hire_date, is_active);

CREATE OR REPLACE TEMP VIEW departments AS
SELECT * FROM VALUES
  ('IT', 'David'),
  ('HR', 'Lisa'),
  ('Sales', 'Ava'),
  ('Finance', 'Lisa')
AS t(department, manager);

CREATE OR REPLACE TEMP VIEW performance AS
SELECT * FROM VALUES
  (1, 92),
  (2, 85),
  (3, 96),
  (4, 88),
  (5, 79),
  (6, 84),
  (7, 70),
  (8, 98),
  (9, 81),
  (10, 90)
AS t(employee_id, performance_score);

SELECT 'employees' AS table_name, COUNT(*) AS row_count FROM employees
UNION ALL
SELECT 'departments', COUNT(*) FROM departments
UNION ALL
SELECT 'performance', COUNT(*) FROM performance;

## 2) Business question: Who is in the organization?

Use a join to combine employee and department information.

In [ ]:
SELECT e.employee_id, e.name, e.department, e.role_title, e.salary, d.manager
FROM employees e
LEFT JOIN departments d
  ON e.department = d.department
ORDER BY e.department, e.salary DESC;

## 3) Business question: Which departments have the highest average salary?

Use `GROUP BY`, aggregation, and ordering to summarize salary data.

In [ ]:
SELECT
  e.department,
  COUNT(*) AS employee_count,
  ROUND(AVG(e.salary), 2) AS avg_salary,
  MAX(e.salary) AS max_salary,
  MIN(e.salary) AS min_salary
FROM employees e
GROUP BY e.department
ORDER BY avg_salary DESC;

## 4) Business question: Which employees are top performers?

Join the employee and performance tables, then rank the team within each department.

In [ ]:
WITH employee_perf AS (
  SELECT e.employee_id, e.name, e.department, e.salary, p.performance_score
  FROM employees e
  JOIN performance p
    ON e.employee_id = p.employee_id
)
SELECT *,
       RANK() OVER (PARTITION BY department ORDER BY performance_score DESC) AS dept_rank
FROM employee_perf
ORDER BY department, dept_rank;

## 5) Business question: How should salary bands be defined?

Use `CASE WHEN` to create salary tiers and recommended bonuses.

In [ ]:
SELECT
  employee_id,
  name,
  department,
  salary,
  CASE
    WHEN salary >= 90000 THEN 'Executive Tier'
    WHEN salary >= 75000 THEN 'Manager Tier'
    WHEN salary >= 65000 THEN 'Professional Tier'
    ELSE 'Entry Tier'
  END AS salary_tier,
  CASE
    WHEN salary >= 90000 THEN 5000
    WHEN salary >= 75000 THEN 3000
    WHEN salary >= 65000 THEN 1500
    ELSE 500
  END AS recommended_bonus
FROM employees
ORDER BY salary DESC;

## 6) Final challenge: executive summary

Build a single summary report that shows key metrics by department.

In [ ]:
WITH dept_summary AS (
  SELECT
    e.department,
    COUNT(*) AS employee_count,
    ROUND(AVG(e.salary), 2) AS avg_salary,
    ROUND(AVG(p.performance_score), 2) AS avg_performance
  FROM employees e
  LEFT JOIN performance p
    ON e.employee_id = p.employee_id
  GROUP BY e.department
),
ranked AS (
  SELECT e.department, e.name, e.salary,
         RANK() OVER (PARTITION BY e.department ORDER BY e.salary DESC) AS salary_rank
  FROM employees e
)
SELECT d.department, d.employee_count, d.avg_salary, d.avg_performance,
       r.name AS top_employee, r.salary AS top_employee_salary
FROM dept_summary d
LEFT JOIN ranked r
  ON d.department = r.department
WHERE r.salary_rank = 1
ORDER BY d.avg_salary DESC;

## Reflection

This project is a realistic example of how SQL is used in analytics and business reporting. The same pattern can be extended to finance, sales, HR, or operations.

Suggested next upgrades:
- add a `projects` table
- add monthly sales or revenue data
- create a reusable view for the department summary
- build a dashboard with KPIs for stakeholders